In [1]:
pip install rank_bm25


In [24]:
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt', quiet=True)

def generate_corpus_bm25_embeddings_with_tokens(corpus):
    """
    Returns list of dicts:
      { "index": i, "text": doc_text, "tokens": [...], "embedding": [...score vector...] }
    'tokens' is the tokenized document (useful for building BM25 later).
    """
    if not corpus:
        return []

    tokenized_corpus = [word_tokenize(doc.lower()) for doc in corpus]
    bm25 = BM25Okapi(tokenized_corpus)

    results = []
    for i, doc_text in enumerate(corpus):
        tokens = tokenized_corpus[i]
        doc_embedding = bm25.get_scores(tokens).tolist()
        results.append({
            "index": i,
            "text": doc_text,
            "tokens": tokens,
            "embedding": doc_embedding
        })
    return results

# Then you can build BM25 directly from item['tokens']:
# tokenized_corpus = [item['tokens'] for item in corpus_embeddings]


Here's an example of how to use the `generate_corpus_bm25_embeddings` function with your existing corpus, and display the output in the requested format:

In [25]:
corpus = [
    "Hello there good man!",
    "It is a beautiful day today.",
    "The weather is nice.",
    "How are you doing today?",
    "This is a test document.",
    "Another example sentence."
]

corpus_embeddings = generate_corpus_bm25_embeddings_with_tokens(corpus)

print("index, text, embedding (first 5 values for brevity):")
for item in corpus_embeddings:
    print(f"{item['index']}, '{item['text']}', {item['embedding'][:5]}...")

index, text, embedding (first 5 values for brevity):
0, 'Hello there good man!', [6.773513187408944, 0.0, 0.0, 0.0, 0.0]...
1, 'It is a beautiful day today.', [0.0, 4.768675822410893, 0.29246457844827434, 0.5646858789452671, 0.8341619752447686]...
2, 'The weather is nice.', [0.0, 0.24983816215621812, 4.35657249089364, 0.0, 0.2694760962995016]...
3, 'How are you doing today?', [0.0, 0.5235346812893369, 0.0, 6.805783182627742, 0.0]...
4, 'This is a test document.', [0.0, 0.773372843445555, 0.29246457844827434, 0.0, 4.578820357454254]...
5, 'Another example sentence.', [0.0, 0.24983816215621812, 0.29246457844827434, 0.0, 0.2694760962995016]...


In [27]:
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt', quiet=True)

def rank_query_from_corpus_embeddings(query, corpus_embeddings, top_k=None):
    """
    Rank `query` against the corpus represented by `corpus_embeddings`.
    This function rebuilds a BM25 index from the original texts stored in corpus_embeddings.
    Returns a sorted list of dicts: [{'rank', 'index', 'text', 'score'}, ...]
    """
    if not corpus_embeddings:
        return []

    # Reconstruct tokenized corpus from the stored texts
    tokenized_corpus = [word_tokenize(item['text'].lower()) for item in corpus_embeddings]

    bm25 = BM25Okapi(tokenized_corpus)

    # Tokenize the incoming query
    query_tokens = word_tokenize(query.lower())

    scores = bm25.get_scores(query_tokens)  # numpy array
    ranked_idx_scores = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)

    if top_k is not None:
        ranked_idx_scores = ranked_idx_scores[:top_k]

    results = []
    for rank, (idx, score) in enumerate(ranked_idx_scores, start=1):
        results.append({
            "rank": rank,
            "index": corpus_embeddings[idx]["index"],
            "text": corpus_embeddings[idx]["text"],
            "score": float(score)
        })

    return results



# Use your existing generator to produce corpus_embeddings (which you already ran)
# corpus_embeddings = generate_corpus_bm25_embeddings(corpus)

# Now rank a query:
query = "Weather is hot"
ranked = rank_query_from_corpus_embeddings(query, corpus_embeddings, top_k=6)
for r in ranked:
    print(f"Rank {r['rank']} | Score {r['score']:.4f} | Text: '{r['text']}'")


Rank 1 | Score 1.3547 | Text: 'The weather is nice.'
Rank 2 | Score 0.0000 | Text: 'Hello there good man!'
Rank 3 | Score 0.0000 | Text: 'It is a beautiful day today.'
Rank 4 | Score 0.0000 | Text: 'How are you doing today?'
Rank 5 | Score 0.0000 | Text: 'This is a test document.'
Rank 6 | Score 0.0000 | Text: 'Another example sentence.'
